# **Elastic Pendulum**
## FEM Implementation using NGSolve
---------------------------------------

The following implementation is a simple example of an elastic pendulum with wall impact using the Finite Element Method (FEM) with NGSolve.
Following references were used to implement the model:
- [Elastic Pendulum - NGS Tutorial 2024](https://docu.ngsolve.org/ngs24/tutorials/00_dynamics.html)
- [Contact Problems - NGS Docu Interactive Tutorial](https://docu.ngsolve.org/latest/i-tutorials/unit-6.2-contact/contact.html)
- [An Interactive Introduction to the Finite Element Method, Joachim Schöberl, TU Wien, ASC](https://jschoeberl.github.io/iFEM/intro.html)

----------------------------------------------

#### **Problem Description**

- Pendulum is fixed at the top and swings under the influence of gravity, initial velocity, and initial angular acceleration (if defined) around the pivot joint (z-axis).
- The Wall is fixed at the top, bottom, and left edges (not shown in the figure).
- The wall is modeled as a linear elastic material (E = 210e9 Pa, nu = 0.3).
- The pendulum can considered with different material model (linear elastic or Neo-Hookean).
- Initial conditions: angular position ($\theta_0$), angular velocity ($\omega_0$), and angular acceleration ($\alpha_0$)
- A contact boundary condition is defined between the pendulum head edge and the right edge of the wall.
- A contact energy is added to the system energy to account for the contact force.

|Pendulum Geometry|Boundary and Initial Condition|
|-----------------|------------------------------|
|![](images/img_dimensions.png)|![](images/img_setup.png)|

---------------------------------------

### **Theory**

#### **1. Modeling Elasticity**

##### **1.1. Kinematics**

|**Property**|**Definition**|
|---|---|
|Body in rest|$\Omega \subset \mathbb{R}^d$|
|Deformation function| $\phi : \Omega \rightarrow {\mathbb R}^3$|
|Displacement| $u(x) = \phi(x) - x$|
|Deformation Gradient| $F = \nabla \phi$|
|Cauchy-Green Strain Tensor| $C = F^T F$|
|Rigid Body Motion| $\phi(x) = a + Qx$, $a \in \mathbb{R}^3$, $Q$ is a rotation matrix |
|Green's Deformation Tensor| $E = \frac{1}{2}(C - I)=\frac{1}{2}\big( \nabla u + \nabla u^T + \nabla u^T \nabla u \big)$|


##### **1.2. Elastic Material Laws**

- When work is applied to a body, it deforms and stores deformation energy. 
- An elastic body returns the work when the external forces are removed.

**Hyperelastic materials**:

Constitutive law which expresses the deformation energy point-wise by the Cauchy-Green strain tensor:
$$
E_{def} = \int_\Omega W(C(u)) \, dx
$$

- Energy density function $W$ may depend on the position $x$ when the material of the body is inhomogeneous (i.e. $W = W(x, C(u))$).

**Isotropic constritution equation**:

- The material properties are the same in all direction.
- The deformation energy is independent of the rotation of the body before deformation.
- Thus, $W$ is a function the (real and positive) eigenvalues $\lambda_i$ of $E$
- **Characteristic polynomial**: $$\det (\lambda I - E) = \lambda^3 - I_1(E) \lambda^2 + I_2(E) \lambda - I_3(E)$$ with the invariants
\begin{align*}
        I_1(E) & =  \operatorname{tr} (E) = \lambda_1 + \lambda_2 + \lambda_3 \\
        I_2(E) & =  \frac{1}{2} [ (\operatorname{tr} E)^2 - \operatorname{tr} (E^2) ] = \lambda_1 \lambda_2 + \lambda_1 \lambda_3 + \lambda_2 \lambda_3 \\
        I_3(E) & =  \det (E) = \lambda_1 \lambda_2 \lambda_3
\end{align*}

- **Rivelin-Ericksen-Theorem** for isotropic materials: $$W(E) = W( \operatorname{tr} (E), \operatorname{tr}(E^2), \det (E) )$$
- Assuming that $W(E=0) = 0$ is a local minimum and $W$ is smooth, we can expand $W$ in a Taylor series around $E=0$:
$$
W(E) = \frac{1}{2} W_{,11} \operatorname{tr} (E)^2 + W_{,2} \operatorname{tr} (E^2) + O(\| E \|^3)
$$

- Dropping higher order terms and Lamé parameters $\lambda := W_{,11}$ (resistance to volumetric deformation) and $\mu := W_{,2}$ (resistance to shear deformation) we obtain the second order energy density (called *Hooke's Law*):
$$
W(E) = \frac{\lambda}{2} \operatorname{tr} (E)^2 + \mu E:E,
$$ 
- It is similar to an elastic spring: The stored energy is $\frac{1}{2} k E^2$, with the spring constant $k$ and elongation $E$.

##### **1.3. Variational Formulation and Equilibrium**

- Let $V$ be a function space of feasible displacements and $\Gamma_D$ the Dirichlet boundary where the displacement is prescribed

- **Total energy = Deformation energy + Potential of external forces**:
$$
J(u) = \int_\Omega W(C(u)) \, dx - \int_\Omega f \cdot u \, dx + \text{Inertia(u)} + \text{Torque(u)}
$$
with 
  - $f$: volume force density
  - Inertia (as a function of displacement $u$, Newmark scheme for $a, v$ updates)
  - Torque is applied as a distributed body force

#### **2. Solving Nonlinear Elasticity Variational Problems**

We consider minimization problems of the form

$$\text{find } u \in V \text{ s.t. } E(u) \leq E(v) \quad \forall~  v \in V.$$

- We are solving this problem using Newton's method.
- We are using the `Variation` integrator of `NGSolve` and formulate the problem through a symbolic description of an energy functional.
- Let $E(u)$ be the energy that is to be minimized for the unknown state $u$.
- A necessary optimality condition is that the derivative at the minimizer $u$ in all directions $v$ vanishes, i.e. 
$$
  \delta E(u) (v) = 0 \quad \forall v \in V
$$

- We assume for our pendulum a Neo-Hookean hyperelastic material model.
- The energy density function is given as
$$
  E(v) := \int_{\Omega} \frac{\mu}{2} ( \operatorname{tr}(F^T F-I)+\frac{2 \mu}{\lambda} \operatorname{det}(F^T F)^{-\frac{\lambda}{2\mu}} - 1) ~~ dx
$$

#### **3. Solving Dynamic Contact Problems**

- Dynamic contact combines for our example nonlinear elasticity with contact constraints in a time-stepping scheme.
- The approach uses a penalty method to enforce the contact conditions and to handle large deformations through the Neo-Hookean material models.

1. **Geometry and Mesh**:
- Contact boundaries must be explicitly defined
- Self-contact is supported (same boundary can contact itself)

2. **Contact Gap Function**:

- Measures the signed distance between the two contact surface
- Negative values indicate penetration

``` python
cf = (X + u-uold - (X.Other() + u.Other() - uold.Other())) * (-specialcf.normal(2).Other())
```
Where 
- `X`: reference position
- `u`: current displacement
- `uold`: displacement from previous time step
- `Other()`: values from the other side of the contact boundary
- `specialcf.normal(2)`: normal vector on the contact boundary (pointing outward from the element)
- `cf`: gap function, negative values indicate penetration

3. **Contact Energy**:

``` python
contact.AddEnergy(IfPos(cf, 1e9*cf*cf, 0), deformed=True)
```
- Penalty method: large stiffness ($1e9$) when surfaces penetrate (cf < 0)
- `IfPos(cf, 1e9*cf*cf, 0)`: adds energy only when `cf` is positive (penetration)
- `deformed=True`: evaluates the energy in the deformed configuration

#### **4. Newmark time-stepping**

References:
- [Newmark-beta method - Wikipedia](https://en.wikipedia.org/wiki/Newmark-beta_method)
- [NGSolve Documentation - Newmark Method](https://docu.ngsolve.org/ngs24/SaS/dynamics_newmark_gen_alpha.html)

The Newmark method is an implicit time-integration scheme for solving second-order differential equations (structural dynamics problems).

1. **Mathematical Foundation**
- The scheme solves the dynamic equilibrium equations by approximating the displacement, velocity, and acceleration at each time step.
$$
M \ddot{u} + C \dot{u} + f^{int} u = f^{ext}
$$
- where:
    - $M$: mass matrix
    - $C$: damping matrix 
    - $f^{int}$: internal force vector
    - $f^{ext}$: external force vector

- New acceleration is obtained from the elasticity operator $K$:
$$
a^{n+1} = f - K(u^{n+1})
$$
- where:
    - Displacement $u^{n+1}$ and the acceleration $a^{n+1}$ at the new time step are unknowns
    - The velocity has to be determined via the time stepping scheme (see below)

2. **Newmark Scheme**

- Trapezoidal rule (average acceleration method) is used to approximate the velocity and acceleration.
- Implicit method: requires solving a nonlinear system at each time step (e.g., using Newton's method).

Key equations of the Newmark method are:

\begin{align}
\frac{u^{n+1}-u^n}{\tau} &= \frac{v^n+v^{n+1}}{2} \\
\frac{v^{n+1}-v^n}{\tau} &= \frac{a^n+a^{n+1}}{2}
\end{align}

They can be rearranged to express $v^{n+1}$ and $a^{n+1}$ in terms of $u^{n+1}$:

\begin{align}
v^{n+1} &= \frac{2}{\tau}(u^{n+1} - u^n) - v^n \\
a^{n+1} &= \frac{2}{\tau}(v^{n+1} - v^n) - a^n
\end{align}

3. **Procedure**

    1. **Prediction:** Start with known values at time step $n$ ($u^n$, $v^n$, $a^n$)
    2. **Implicit Solution:** Solve nonlinear system for $u^{n+1}$ using Newton's method.
    3. **Correction:** Update $v^{n+1}$ and $a^{n+1}$ using the equations above.
    4. **Advance:** Move to the next time step and repeat.

#### **5. Constraint Implementation - Pendulum Pivot**

```python
bfa += (InnerProduct(u, p) + InnerProduct(v, q)) * ds('top')
```

- Lagrange multipliers `p` and `q` are introduced to enforce the constraints at the pendulum pivot.
- Mean value constraint: Controls the average displacement of pivot point.
- Allows the pendulum to swing freely while keeping the pivot fixed.

#### **6. Variational Form - Weak Formulation**

- **Principle of Virtual Work**: The weak form of the equilibrium equations is derived from the principle of virtual work, which states that the work done by internal forces equals the work done by external forces for any virtual displacement
$$
\delta W_{\text{internal}} = \delta W_{\text{external}}
$$

- `Variation(NeoHooke(C(u))*dx).Compile()`: Computes the variation of the Neo-Hookean energy density integrated over the domain. Corresponds to the internal elastic work from material nonlinearity.
- `acc_new*v*dx`: Represents the inertial forces due to acceleration.
- `-force*v*dx`: Represents the work done by external forces (e.g., gravity).


#### **7. Key Points**

1. **Large Deformations**: \
The Neo-Hookean material model captures large deformations and nonlinear elasticity.

2. **Contact Handling**: \
The penalty method effectively enforces contact constraints, preventing interpenetration.

3. **Time Integration**: \
The Newmark method provides a stable and accurate time-stepping scheme for dynamic simulations.

4. **Constraint Enforcement**: \
Lagrange multipliers are used to enforce constraints at the pendulum pivot, allowing for realistic motion.

5. **Variational Formulation**: \
The weak form derived from the principle of virtual work ensures that the internal and external forces are balanced for any virtual displacement.

--------------------------------------